# DCIT 313 | Lecture 2 Student Notebook
## Problem solving and heuristic search
**Scenario:** synthetic directed route graph; costs are arbitrary units and are **not** real-world travel distances.

**Competencies:** formulate states/actions/costs; implement BFS/UCS/A*; test and critique heuristic design.

**Tools:** Python 3 standard library only. Keep this `.ipynb` beside the supplied CSV and `.txt` files. Complete the `TODO` blocks rather than copying the TA solution.

In [1]:
from collections import deque
from pathlib import Path
import csv, heapq, math

HERE = Path.cwd()
print("Working folder:", HERE)

Working folder: C:\Users\SoarMediaPC\.gemini\antigravity\scratch\DCIT313-Artificial-Intelligence\Lab2-Problem-Solving-and-Heuristic-Search


### 0. Load and inspect the data
We use a **directed** graph. Do not infer that an edge can be used in reverse. Sort successors alphabetically for deterministic traces. The goal test occurs when a node is **popped**, not when first generated.

In [3]:
graph = {}
for row in csv.DictReader(open("synthetic_route_edges.csv", newline="", encoding="utf-8")):
    graph.setdefault(row["source"], []).append((row["destination"], float(row["cost_units"])))
for _, edges in list(graph.items()):
    for v, _ in edges: graph.setdefault(v, [])
for u in graph: graph[u] = sorted(graph[u], key=lambda e: e[0])
h = {r["node"]:float(r["heuristic_remaining_cost"])
     for r in csv.DictReader(open("synthetic_heuristics.csv", newline="", encoding="utf-8"))}
print("States:", sorted(graph))
print("Edges:", sum(len(v) for v in graph.values()))
print("h:", h)

States: ['A', 'B', 'C', 'D', 'E', 'G', 'S']
Edges: 11
h: {'S': 8.0, 'A': 8.0, 'B': 4.0, 'C': 6.0, 'D': 4.0, 'E': 3.0, 'G': 0.0}


### Part A — Search Problem Formulation (Formal Specification)

#### 1. Formal 5-Tuple Formulation
- **State Space ($S$):** The set of discrete network junctions: $S = \{S, A, B, C, D, E, G\}$.
- **Initial State ($s_0$):** $S$ (Campus Gate).
- **Action Set ($A(s)$):** Legal directed transitions available from node $s$:
  - $A(S) = \{A, B\}$
  - $A(A) = \{C, D\}$
  - $A(B) = \{D, E\}$
  - $A(C) = \{D, G\}$
  - $A(D) = \{E, G\}$
  - $A(E) = \{G\}$
  - $A(G) = \emptyset$ (Terminal State)
- **Transition Model ($T(s, a)$):** Deterministic successor state resulting from action $a$ taken at state $s$, i.e., $T(s, a) = a$.
- **Goal Test ($\text{Goal}(s)$):** Returns `True` if $s = G$ (District Hospital), `False` otherwise.
- **Step-Cost Function ($c(s, a, s')$):** Non-negative edge weights specified in `synthetic_route_edges.csv` (e.g. $c(S, A) = 2.0, c(S, B) = 4.0$).

#### 2. Why 'Current Node' is a Sufficient State Representation
In this synthetic planning problem, the environment is **static, fully observable, and deterministic**. The future cost to reach the goal depends exclusively on the current junction location, not on previous path history (Markovian property). There are no vehicle capacity limits, time-varying traffic bottlenecks, or dynamic battery depletion constraints.

#### 3. Real-World Variables Requiring Richer State Space
1. **Time-Dependent Congestion / Travel Delays:** Edge costs vary dynamically by time of day; the state would need to be augmented as $(s, t)$ where $t$ is the elapsed arrival time.
2. **Vehicle Resource / State of Charge (SoC):** For electric campus shuttles, battery depletion or charging constraints require state $(s, \text{battery\_remaining})$.

#### 4. Hop Count vs. Path Cost
**Prediction:** A route with fewer hops does **not** necessarily have the least total weighted cost. BFS finds the minimum number of edge traversals (e.g., $S \to A \to C \to G$ has 3 hops but a high total cost of $2 + 2 + 10 = 14$), whereas UCS finds a 4-hop route ($S \to B \to D \to E \to G$) with a much lower total cost of $4 + 1 + 1 + 3 = 9$.

### Part B — Implement BFS (20 minutes)
**Contract:** `bfs(graph, start, goal)` returns `(path, cost, expanded)`; `path=None` on failure. `expanded` is the sequence of popped nodes, including the goal. Mark a state discovered **when enqueued**. Use a `deque` and alphabetical successors. Do not use `networkx` or an LLM as a substitute for implementation.

In [6]:
def path_cost(graph, path):
    if path is None:
        return math.inf
    return sum(dict(graph[u])[v] for u, v in zip(path, path[1:]))

def bfs(graph, start="S", goal="G"):
    """FIFO graph search returning (path, cost, expanded).
    - Discovered on enqueue (visited set)
    - Successors explored in alphabetical order
    - Goal test evaluated when node is popped
    """
    if start == goal:
        return [start], 0.0, [start]
    
    discovered = {start: None}
    queue = deque([start])
    expanded = []
    
    while queue:
        u = queue.popleft()
        expanded.append(u)
        if u == goal:
            path = []
            curr = u
            while curr is not None:
                path.append(curr)
                curr = discovered[curr]
            path.reverse()
            return path, path_cost(graph, path), expanded
        
        for v, _ in graph.get(u, []):
            if v not in discovered:
                discovered[v] = u
                queue.append(v)
                
    return None, math.inf, expanded


After implementing BFS, run this check. Record the path and total weighted cost. Explain why minimizing the number of edges is not the same as minimizing route cost.

In [8]:
result_bfs = bfs(graph)
print("BFS:", result_bfs)
assert result_bfs[0][0] == "S" and result_bfs[0][-1] == "G"
assert result_bfs[1] == path_cost(graph, result_bfs[0])

BFS: (['S', 'A', 'C', 'G'], 14.0, ['S', 'A', 'B', 'C', 'D', 'E', 'G'])


### Part C — UCS and greedy best-first (30 minutes)
Use `heapq`. A graph state may first be discovered by a **more expensive** route; update `best_g` and `parent` only if the new path is strictly cheaper. Since Python heaps do not have decrease-key, allow duplicate heap entries and skip a popped entry when its stored `g` is not the current best known value.

UCS priority = `g`. Greedy priority = `h`. A* priority = `(g+h, h, node)`. For UCS ties use the node label; for greedy ties use the label.

In [10]:
def best_first(graph, h, start="S", goal="G", mode="ucs"):
    """Priority search implementation supporting UCS, Greedy, and A*.
    - Validates non-negative edge costs
    - Uses min-heap with stale entry handling
    - Priority ordering:
        UCS:    (g, node)
        Greedy: (h, node)
        A*:     (g + h, h, node)
    - Goal test evaluated upon POP
    """
    # Validate non-negative edge costs
    for u in graph:
        for v, cost in graph[u]:
            if cost < 0:
                raise ValueError(f"Negative edge cost detected on ({u}, {v}): {cost}")
                
    best_g = {start: 0.0}
    parent = {start: None}
    
    def make_priority(node, g_val):
        h_val = h.get(node, 0.0)
        if mode == "ucs":
            return (g_val, node)
        elif mode == "greedy":
            return (h_val, node)
        elif mode == "astar":
            return (g_val + h_val, h_val, node)
        else:
            raise ValueError(f"Unsupported mode: {mode}")
            
    heap = []
    prio = make_priority(start, 0.0)
    heapq.heappush(heap, (*prio, start, 0.0))
    
    expanded = []
    closed = set() # For greedy best-first to prevent duplicate cycle exploration
    
    while heap:
        item = heapq.heappop(heap)
        node = item[-2]
        stored_g = item[-1]
        
        # Stale entry check: if stored_g is worse than best_g, discard
        if stored_g > best_g[node]:
            continue
            
        if mode == "greedy":
            if node in closed:
                continue
            closed.add(node)
            
        expanded.append(node)
        
        if node == goal:
            path = []
            curr = node
            while curr is not None:
                path.append(curr)
                curr = parent[curr]
            path.reverse()
            return path, path_cost(graph, path), expanded
            
        for v, cost in graph.get(node, []):
            cand_g = stored_g + cost
            if cand_g < best_g.get(v, math.inf):
                best_g[v] = cand_g
                parent[v] = node
                prio_v = make_priority(v, cand_g)
                heapq.heappush(heap, (*prio_v, v, cand_g))
                
    return None, math.inf, expanded


In [11]:
ucs_path, ucs_cost, ucs_expanded = best_first(graph,h,mode="ucs")
print("UCS:", ucs_path,ucs_cost,ucs_expanded)
assert ucs_cost == path_cost(graph, ucs_path)

UCS: ['S', 'B', 'D', 'E', 'G'] 9.0 ['S', 'A', 'B', 'C', 'D', 'E', 'G']


### Part D — Add A* and verify the heuristic (25 minutes)
**Admissibility:** `0 <= h(u) <= h_star(u)`, where `h_star` is the actual least remaining cost. In this small graph you can compute `h_star` by running UCS from each state.
**Consistency:** `h(u) <= c(u,v) + h(v)` on every directed edge; also `h(goal)=0`. A* with a consistent heuristic and nonnegative edges is guaranteed optimal on our finite graph. Test a modified `h[B]=7` and identify violations.

In [13]:
astar_path, astar_cost, astar_expanded = best_first(graph,h,mode="astar")
print("A*:",astar_path,astar_cost,astar_expanded)
assert astar_cost == ucs_cost  # compare independently implemented algorithms

A*: ['S', 'B', 'D', 'E', 'G'] 9.0 ['S', 'B', 'D', 'E', 'G']


In [14]:
def consistency_violations(graph, h, goal="G"):
    """Checks consistency/monotonicity: h(u) <= c(u, v) + h(v) for all edges, and h(goal) == 0.
    Returns list of violating tuples (u, v, cost).
    """
    violations = []
    if h.get(goal, 0.0) != 0.0:
        violations.append(("GOAL_CHECK", goal, h.get(goal)))
    for u in sorted(graph):
        for v, cost in graph[u]:
            if h.get(u, 0.0) > cost + h.get(v, 0.0):
                violations.append((u, v, cost))
    return violations

print("Original heuristic consistency violations:", consistency_violations(graph, h))
altered = dict(h)
altered["B"] = 7.0
print("Altered heuristic (h[B]=7) violations:", consistency_violations(graph, altered))


Original heuristic consistency violations: []
Altered heuristic (h[B]=7) violations: [('B', 'D', 1.0)]


### Part E — A* on a small grid (20 minutes)
Read `grid_map.txt` (6×6). `S` is the start, `G` the destination, `#` blocked. Allowed moves are four-neighbour (up/down/left/right), cost 1. Compute Manhattan distance from any cell to `G` and run A*. Do **not** allow diagonal movement.

In [16]:
grid = Path("grid_map.txt").read_text().strip().splitlines()
print("Loaded Grid Map:")
for row in grid:
    print(row)

def grid_to_graph(grid):
    """Converts 2D ASCII grid into adjacency graph, start coordinate, and goal coordinate."""
    rows = len(grid)
    cols = len(grid[0])
    start = None
    goal = None
    adj = {}
    
    for r in range(rows):
        for c in range(cols):
            ch = grid[r][c]
            if ch == "S":
                start = (r, c)
            elif ch == "G":
                goal = (r, c)
            if ch != "#":
                adj[(r, c)] = []
                
    # Four-neighbour movements: Up, Down, Left, Right with unit cost 1.0
    for (r, c) in adj:
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] != "#":
                adj[(r, c)].append(((nr, nc), 1.0))
        adj[(r, c)].sort(key=lambda edge: edge[0])
        
    return adj, start, goal

adj_grid, start_pos, goal_pos = grid_to_graph(grid)
# Manhattan distance heuristic: |r - r_goal| + |c - c_goal|
h_grid = {pos: float(abs(pos[0] - goal_pos[0]) + abs(pos[1] - goal_pos[1])) for pos in adj_grid}

grid_path, grid_cost, grid_expanded = best_first(adj_grid, h_grid, start=start_pos, goal=goal_pos, mode="astar")
print("\nGrid A* Results:")
print("Start:", start_pos, "Goal:", goal_pos)
print("Optimal Path:", grid_path)
print("Total Cost:", grid_cost)
print("Nodes Expanded:", len(grid_expanded))


Loaded Grid Map:
S.....
.###..
...#..
.#....
.#.##.
.....G

Grid A* Results:
Start: (0, 0) Goal: (5, 5)
Optimal Path: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (1, 5), (2, 5), (3, 5), (4, 5), (5, 5)]
Total Cost: 10.0
Nodes Expanded: 11


### Part F — Robustness Checks and Technical Discussion

#### 1. Performance Summary Table
| Algorithm | Returned Route | Total Cost | Nodes Expanded | Optimality |
| :--- | :--- | :---: | :---: | :---: |
| **BFS** | `['S', 'A', 'C', 'G']` | 14.0 | 7 nodes (`['S', 'A', 'B', 'C', 'D', 'E', 'G']`) | Fewest hops (3), Suboptimal cost |
| **UCS** | `['S', 'B', 'D', 'E', 'G']` | 9.0 | 7 nodes (`['S', 'A', 'B', 'C', 'D', 'E', 'G']`) | Cost-optimal (9.0) |
| **Greedy** | `['S', 'B', 'E', 'G']` | 11.0 | 4 nodes (`['S', 'B', 'E', 'G']`) | Fast (4 expansions), Suboptimal cost |
| **A\*** | `['S', 'B', 'D', 'E', 'G']` | 9.0 | 5 nodes (`['S', 'B', 'D', 'E', 'G']`) | Cost-optimal (9.0), Minimal expansions |

#### 2. Key Algorithmic Insights
- **BFS vs. UCS:** BFS optimizes path depth (number of edges). It treats every edge as unit cost. Because edge weights in our referral graph vary from 1 to 10, BFS picks the 3-hop path with high cost 14. UCS prioritizes accumulated path cost $g(n)$, correctly discovering the cheaper 4-hop path with cost 9.
- **Greedy Best-First vs. A\*:** Greedy search relies solely on $h(n)$ to aggressively dive towards the goal, expanding only 4 nodes but settling for suboptimal cost 11 ($S \to B \to E \to G$). A* combines past cost and estimated future cost ($f = g + h$), pruning wasteful branches while strictly preserving optimality.
- **Consistent vs. Inadmissible Heuristic:** An admissible heuristic never overestimates $h^*(n)$. Consistency ($h(u) \le c(u,v) + h(v)$) guarantees monotonic non-decreasing $f(n)$ values, ensuring that the first time a goal node is popped from the priority queue, its path is guaranteed optimal without needing to reopen closed nodes.

#### 3. Negative Edge Costs
UCS and A* fundamentally assume that edge costs are non-negative ($c(u, v) \ge 0$). This ensures that accumulated path cost $g(n)$ is non-decreasing along any path, so popping a node guarantees the cheapest route to that state has been settled. Negative edges violate this monotonic property and can create infinite negative cycles, rendering Dijkstra-style priority queue algorithms invalid (requiring Bellman-Ford or SPFA instead).

### Extension — Depth-First Search (DFS) Implementation & AI Coding Critique

#### Potential AI Assistant Pitfalls Detected by Tests
1. **Goal Test on Generation vs. Expansion:** A frequent bug generated by AI assistants in UCS/A* is terminating when the goal is first *generated* (enqueued) rather than when it is *popped*. On our graph, $G$ is first generated via $C \to G$ with cost 14. If the algorithm stopped immediately on generation, it would return cost 14 instead of the optimal 9 ($S \to B \to D \to E \to G$). Our test `assert astar_cost == 9.0` immediately flags this defect.
2. **Inefficient FIFO with `list.pop(0)`:** Using Python's built-in list for a FIFO queue results in $O(N)$ pop operations, degrading BFS to $O(N^2)$ time complexity. We enforce `collections.deque.popleft()` for guaranteed $O(1)$ amortized operations.
3. **Closed-Set Premature Termination:** In graphs with multiple routes, permanently locking nodes into a closed set without checking if a cheaper $g(n)$ path is found prevents path relaxation.

In [19]:
# Robustness Tests: Disconnected Graph, Zero-Cost Edge, and Negative Cost Handling
print('--- Robustness Test 1: Unreachable Goal ---')
disconnected_graph = {
    'S': [('A', 2.0)],
    'A': [('B', 3.0)],
    'B': [],
    'G': []
}
h_disc = {'S': 5.0, 'A': 3.0, 'B': 1.0, 'G': 0.0}
res_disc, cost_disc, exp_disc = best_first(disconnected_graph, h_disc, start='S', goal='G', mode='astar')
print('Disconnected Result:', res_disc, 'Cost:', cost_disc, 'Expanded:', exp_disc)
assert res_disc is None and cost_disc == math.inf

print('\n--- Robustness Test 2: Zero-Cost Edges ---')
zero_cost_graph = {
    'S': [('A', 0.0)],
    'A': [('G', 2.0)],
    'G': []
}
h_zero = {'S': 2.0, 'A': 2.0, 'G': 0.0}
res_zero, cost_zero, exp_zero = best_first(zero_cost_graph, h_zero, start='S', goal='G', mode='astar')
print('Zero-cost Result:', res_zero, 'Cost:', cost_zero, 'Expanded:', exp_zero)
assert res_zero == ['S', 'A', 'G'] and cost_zero == 2.0

print('\n--- Robustness Test 3: Negative Cost Rejection ---')
neg_graph = {'S': [('A', -2.0)], 'A': [('G', 1.0)], 'G': []}
try:
    best_first(neg_graph, h_zero, start='S', goal='G', mode='ucs')
except ValueError as e:
    print('Successfully caught negative edge error:', e)


--- Robustness Test 1: Unreachable Goal ---
Disconnected Result: None Cost: inf Expanded: ['S', 'A', 'B']

--- Robustness Test 2: Zero-Cost Edges ---
Zero-cost Result: ['S', 'A', 'G'] Cost: 2.0 Expanded: ['S', 'A', 'G']

--- Robustness Test 3: Negative Cost Rejection ---
Successfully caught negative edge error: Negative edge cost detected on (S, A): -2.0


In [20]:
def dfs(graph, start="S", goal="G"):
    """LIFO depth-first search using an explicit stack."""
    stack = [(start, [start])]
    visited = set()
    expanded = []
    
    while stack:
        node, path = stack.pop()
        if node in visited:
            continue
        visited.add(node)
        expanded.append(node)
        if node == goal:
            return path, path_cost(graph, path), expanded
        # Reversed alphabetical push to preserve alphabetical pop order
        for nxt, _ in sorted(graph.get(node, []), key=lambda e: e[0], reverse=True):
            if nxt not in visited:
                stack.append((nxt, path + [nxt]))
                
    return None, math.inf, expanded

dfs_path, dfs_cost, dfs_expanded = dfs(graph)
print("DFS Path:", dfs_path, "Cost:", dfs_cost, "Expanded:", dfs_expanded)


DFS Path: ['S', 'A', 'C', 'D', 'E', 'G'] Cost: 10.0 Expanded: ['S', 'A', 'C', 'D', 'E', 'G']
